# Train the Card Model (run in Google Colab)

Trains a YOLO11n model to read card **rank** (A, 2-10, J, Q, K) and face-down cards (`DOWN`) from the robot's camera, then exports it to ONNX for the Jetson.

**Before you start:** *Runtime > Change runtime type > T4 GPU*.

Steps:
1. Install, upload our `card_dataset` (from `card_data_collector.ipynb`)
2. (Optional) Stage 1: pretrain on a public playing-card dataset
3. Stage 2: fine-tune on our own pictures
4. Check accuracy (goal from the proposal: 90%)
5. Export `best.onnx` and download it

In [ ]:
!pip install -q ultralytics roboflow

import os, glob, random, shutil, zipfile
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

CLASSES = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'DOWN']   # same order as the collector, DOWN = face-down card
IMGSZ = 416   # small enough for the Jetson Nano
os.chdir('/content')

## 1. Upload our dataset
On the Jetson, zip the folder first (in a Jupyter terminal, inside `HackRPI26/`):
```
zip -r card_dataset.zip card_dataset
```
Download that zip to your laptop, then upload it with the cell below.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick card_dataset.zip

with zipfile.ZipFile('card_dataset.zip') as z:
    z.extractall('/content')
print("Images:", len(glob.glob('/content/card_dataset/images/*.jpg')))

## Split ours into train / val (80 / 20)
Val pictures are never trained on, so the accuracy number at the end is honest.

In [ ]:
def split_dataset(src, dst, val_frac=0.2, seed=0):
    names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{src}/images/*.jpg'))
    random.Random(seed).shuffle(names)
    n_val = max(1, int(len(names) * val_frac))
    for split, group in [('val', names[:n_val]), ('train', names[n_val:])]:
        os.makedirs(f'{dst}/{split}/images', exist_ok=True)
        os.makedirs(f'{dst}/{split}/labels', exist_ok=True)
        for n in group:
            shutil.copy(f'{src}/images/{n}.jpg', f'{dst}/{split}/images/')
            shutil.copy(f'{src}/labels/{n}.txt', f'{dst}/{split}/labels/')
    return len(names) - n_val, n_val


def write_yaml(path, root):
    with open(path, 'w') as f:
        f.write(f"path: {root}\ntrain: train/images\nval: val/images\n")
        f.write(f"nc: {len(CLASSES)}\nnames: {CLASSES}\n")


shutil.rmtree('/content/ours', ignore_errors=True)
n_train, n_val = split_dataset('/content/card_dataset', '/content/ours')
write_yaml('/content/ours.yaml', '/content/ours')
print("train:", n_train, " val:", n_val)

## 2. (Optional) Stage 1: pretrain on a public dataset
Gives the model a head start on what cards look like. Skip this cell if you don't want to.

1. Make a free account at [roboflow.com](https://roboflow.com) and copy your API key (Settings > API Keys). **Paste it yourself, don't commit it to git.**
2. Pick a playing-card dataset on [Roboflow Universe](https://universe.roboflow.com/search?q=playing%20cards) (search "playing cards", 52 classes). On its page click *Download > YOLOv8 > show download code* and copy the `workspace`, `project`, and `version` into the cell.

Public datasets label 52 cards (like `KH`, `10S`). The cell converts them to our 13 ranks by dropping the suit. Public sets have no face-down cards, so `DOWN` is learned only from our pictures in stage 2: collect plenty of them.

In [ ]:
USE_PUBLIC = True
ROBOFLOW_API_KEY = ''            # paste your key here (don't commit it)
WORKSPACE = 'augmented-startups' # from the dataset's download code
PROJECT = 'playing-cards-ow27d'
VERSION = 4


def rank_of(name):
    """'KH' -> 'K', '10S' -> '10', 'AC' -> 'A'. Returns None if it isn't a card."""
    r = str(name).strip().upper()[:-1]
    return r if r in CLASSES else None


def remap_labels(label_dir, old_names):
    old_to_new = {i: (CLASSES.index(rank_of(n)) if rank_of(n) else None) for i, n in enumerate(old_names)}
    for path in glob.glob(f'{label_dir}/*.txt'):
        out = []
        for line in open(path).read().splitlines():
            parts = line.split()
            if len(parts) == 5 and old_to_new.get(int(parts[0])) is not None:
                out.append(' '.join([str(old_to_new[int(parts[0])])] + parts[1:]))
        open(path, 'w').write('\n'.join(out) + ('\n' if out else ''))


if USE_PUBLIC:
    import yaml
    from roboflow import Roboflow
    ds = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8', location='/content/public')
    old_names = yaml.safe_load(open('/content/public/data.yaml'))['names']
    if isinstance(old_names, dict):
        old_names = [old_names[k] for k in sorted(old_names)]
    print("Unmapped classes:", [n for n in old_names if rank_of(n) is None])
    for split in ['train', 'valid', 'test']:
        if os.path.isdir(f'/content/public/{split}/labels'):
            remap_labels(f'/content/public/{split}/labels', old_names)
    with open('/content/public.yaml', 'w') as f:
        f.write("path: /content/public\ntrain: train/images\nval: valid/images\n")
        f.write(f"nc: {len(CLASSES)}\nnames: {CLASSES}\n")

    model = YOLO('yolo11n.pt')
    model.train(data='/content/public.yaml', epochs=30, imgsz=IMGSZ, batch=32,
                fliplr=0.0,   # mirrored cards aren't real cards
                project='/content/runs', name='public', exist_ok=True)
    START_WEIGHTS = '/content/runs/public/weights/best.pt'
else:
    START_WEIGHTS = 'yolo11n.pt'
print("Stage 2 starts from:", START_WEIGHTS)

## 3. Stage 2: fine-tune on our pictures
If you skipped stage 1, set `START_WEIGHTS = 'yolo11n.pt'` (the cell above does this when `USE_PUBLIC = False`).

In [ ]:
model = YOLO(START_WEIGHTS)
model.train(data='/content/ours.yaml', epochs=100, imgsz=IMGSZ, batch=16,
            fliplr=0.0, degrees=10, patience=25,
            project='/content/runs', name='ours', exist_ok=True)
BEST = '/content/runs/ours/weights/best.pt'

## 4. Check accuracy
Card reading accuracy on the val pictures = the model's most confident box has the right rank.
Proposal goal: **at least 90%**.

In [ ]:
model = YOLO(BEST)
model.val(data='/content/ours.yaml', imgsz=IMGSZ)

right, total, wrong = 0, 0, []
for img in sorted(glob.glob('/content/ours/val/images/*.jpg')):
    true = CLASSES[int(open(img.replace('/images/', '/labels/').replace('.jpg', '.txt')).read().split()[0])]
    r = model(img, imgsz=IMGSZ, verbose=False)[0]
    pred = CLASSES[int(r.boxes.cls[r.boxes.conf.argmax()])] if len(r.boxes) else 'none'
    total += 1
    if pred == true:
        right += 1
    else:
        wrong.append((img, true, pred))

print(f"Card reading accuracy: {right}/{total} = {100*right/max(total,1):.1f}%")
for img, true, pred in wrong:
    print(f"  {os.path.basename(img)}: true {true}, predicted {pred}")

## See some predictions

In [ ]:
imgs = random.sample(glob.glob('/content/ours/val/images/*.jpg'), min(6, n_val))
plt.figure(figsize=(15, 8))
for i, img in enumerate(imgs):
    r = model(img, imgsz=IMGSZ, verbose=False)[0]
    plt.subplot(2, 3, i + 1)
    plt.imshow(r.plot()[:, :, ::-1])
    plt.axis('off')
plt.show()

## 5. Export for the Jetson
ONNX with opset 12 so the Jetson's older software can load it. Downloads `best.pt` and `best.onnx`; put them in `HackRPI26/models/` on the Jetson.

In [ ]:
onnx_path = YOLO(BEST).export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print("Exported:", onnx_path)
files.download(BEST)
files.download(onnx_path)